In [2]:
from process_framework.composition.retries import RetryArgs
from process_framework.pipeline.settings import SettingsBase

In [3]:
# retries as top-level settings
class SettingsWithRetries(RetryArgs, SettingsBase):
    index:str

s = SettingsWithRetries(
    index='a',
    attempts=2,
    wait_seconds=2,
    wait_exponent=2,
    wait_jitter=1.0,
    log_error_level=None,
    log_before_level=None
)

(s.get_wait(), s.get_stop(), s.get_before(), s.get_retry_error_callback)

(<tenacity.wait.wait_exponential_jitter at 0x22c71bdfd40>,
 None,
 <bound method RetryArgs.get_retry_error_callback of SettingsWithRetries(attempts=2, wait_seconds=2, wait_exponent=2, wait_jitter=1.0, log_error_level=None, log_before_level=None, index='a')>)

In [4]:
# a retry step
from process_framework.steps.retry_step import Retry
from process_framework._testing.steps.random_failure import RandomFail
from process_framework._testing.steps.assign_from_func import AssignFromFunc
from process_framework import Reference
import logging

ref = Reference(list)

inner = AssignFromFunc(
    output_=ref,
    func = lambda : ['A', 'B', 'C', 'D']
)

inner.do()

print(ref)

ref.set_value(None)

outer = Retry(
    step=RandomFail(
        step=inner,
        gte=9
    ),
    retry=RetryArgs(
        attempts=3,
        log_before_level=logging.WARNING,
        log_error_level=logging.WARNING
    )
)

outer.do()

print(ref)

Reference(_type=<class 'list'>, value=['A', 'B', 'C', 'D'])
Reference(_type=<class 'list'>, value=None)


In [ ]:
lists = []
for i in range(10):
    lists += (list(range(i, i + 20)))
    if (i % 8) == 0:
        lists.append(None)

from typing import Sequence
from process_framework.steps.batch_processing_step import BatchProcessor
from process_framework.steps.logging_step import LogInput
from process_framework.steps import ModifyingStep

class AddOne(ModifyingStep[Sequence]):
    def transform_value(self, input_: Sequence) -> Sequence:
        return [t + 1 for t in input_]
    

lr = Reference(list, lists)
b = Reference(Sequence)

bp = BatchProcessor(
    input_=lr,
    steps=[LogInput(b, level=logging.WARNING), AddOne(b, b)],
    batch=b,
    batch_size=3,
    retry=RetryArgs(attempts=3, log_before_level=logging.WARNING, log_error_level=logging.WARNING)
)

bp.do()